# Embedding Composition Analysis

How token and positional embeddings compose to form
the initial residual stream: norm balance, alignment,
pairwise similarity, and effective dimensionality.

## Why This Matters

Composition embedding measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.embedding_composition_analysis import (
    token_position_balance, token_position_alignment,
    combined_embedding_properties, embedding_subspace_analysis,
    embedding_composition_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Token vs Position Balance

Which contributes more to the initial representation: token identity or position?

In [ ]:
result = token_position_balance(model, tokens)
print(f"Dominant: {result['dominant']} (mean token fraction: {result['mean_token_fraction']:.4f})")
for p in result['per_position']:
    tok_bar = '█' * int(p['token_fraction'] * 20)
    pos_bar = '█' * int((1 - p['token_fraction']) * 20)
    print(f"  Pos {p['position']}: tok={p['token_norm']:.4f} {tok_bar}")
    print(f"          pos={p['position_norm']:.4f} {pos_bar}")

## Token-Position Alignment

Do token and positional embeddings point in similar directions?
High alignment means redundant; low alignment means complementary information.

In [ ]:
result = token_position_alignment(model, tokens)
print(f"Mean alignment: {result['mean_alignment']:.4f} (is_aligned={result['is_aligned']})")
for p in result['per_position']:
    bar_len = int(abs(p['cosine']) * 20)
    sign = '+' if p['cosine'] > 0 else '-'
    print(f"  Pos {p['position']}: cos={p['cosine']:+.4f} {sign}{'█' * bar_len}")

## Combined Embedding Properties

Pairwise similarity of the combined (token + position) embeddings
across positions. High similarity = representations start similar.

In [ ]:
result = combined_embedding_properties(model, tokens)
print(f"Mean pairwise similarity: {result['mean_pairwise_similarity']:.4f}")
print(f"Mean norm: {result['mean_norm']:.4f}")
for p in result['per_position']:
    bar = '█' * int(p['combined_norm'] * 10)
    print(f"  Pos {p['position']}: norm={p['combined_norm']:.4f} {bar}")

## Embedding Subspace Analysis

Effective dimensionality of the initial embeddings via SVD.

In [ ]:
result = embedding_subspace_analysis(model, tokens)
print(f"Effective rank: {result['effective_rank']:.2f} / {result['n_tokens']} tokens")
print(f"Dims for 90% variance: {result['dim_for_90_pct']}")
print(f"Top singular value: {result['top_singular_value']:.4f}")

## Composition Summary

In [ ]:
result = embedding_composition_summary(model, tokens)
for k, v in result.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")